# 🩺 Mediosis — Colab runner
**RxDecode** (prescription OCR → BiLSTM-CRF → fastText drug correction → interactions, EN/हिन्दी) + **XCheck** (chest X-ray DenseNet + Grad-CAM vs report NLP with BiGRU self-attention).

1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Open the `gradio.live` link printed in step 5
4. Step 6 pushes everything to your GitHub

> ⚠️ Research/education demo — not a medical device.

In [ ]:
#@title 0 · GPU check
!nvidia-smi -L || echo 'No GPU — still works on CPU, just slower'

## 1 · Write the project files

In [ ]:
import os
os.makedirs('/content/mediosis/mediosis', exist_ok=True)
os.makedirs('/content/mediosis/samples', exist_ok=True)
%cd /content/mediosis

In [ ]:
%%writefile mediosis/__init__.py
"""Mediosis — prescription decoding + radiology second-reader."""


In [ ]:
%%writefile mediosis/lexicon.py
"""Curated drug lexicon, interaction rules and Rx shorthand for Mediosis RxDecode.

Screening-level knowledge base only. NOT exhaustive. Always verify with a pharmacist.
"""

# brand / generic token -> list of generic ingredients
BRANDS = {
    # analgesics / antipyretics / NSAIDs
    "crocin": ["paracetamol"], "dolo": ["paracetamol"], "calpol": ["paracetamol"],
    "combiflam": ["ibuprofen", "paracetamol"], "brufen": ["ibuprofen"],
    "voveran": ["diclofenac"], "zerodol": ["aceclofenac"], "ecosprin": ["aspirin"],
    # antibiotics
    "augmentin": ["amoxicillin", "clavulanic acid"], "mox": ["amoxicillin"],
    "azithral": ["azithromycin"], "azee": ["azithromycin"], "ciplox": ["ciprofloxacin"],
    "zifi": ["cefixime"], "metrogyl": ["metronidazole"], "flagyl": ["metronidazole"],
    "claribid": ["clarithromycin"],
    # GI
    "pantocid": ["pantoprazole"], "omez": ["omeprazole"], "razo": ["rabeprazole"],
    "digene": ["antacid (aluminium/magnesium hydroxide)"], "ondem": ["ondansetron"],
    "emeset": ["ondansetron"], "domstal": ["domperidone"],
    # allergy / respiratory
    "allegra": ["fexofenadine"], "cetzine": ["cetirizine"], "okacet": ["cetirizine"],
    "levocet": ["levocetirizine"], "montair": ["montelukast"], "asthalin": ["salbutamol"],
    "deriphyllin": ["etofylline", "theophylline"], "wysolone": ["prednisolone"],
    # diabetes / cardio
    "glycomet": ["metformin"], "amaryl": ["glimepiride"], "januvia": ["sitagliptin"],
    "telma": ["telmisartan"], "losar": ["losartan"], "amlong": ["amlodipine"],
    "amlokind": ["amlodipine"], "clopilet": ["clopidogrel"], "atorva": ["atorvastatin"],
    "rosuvas": ["rosuvastatin"], "lasix": ["furosemide"], "aldactone": ["spironolactone"],
    "lanoxin": ["digoxin"], "warf": ["warfarin"],
    # thyroid / supplements
    "thyronorm": ["levothyroxine"], "eltroxin": ["levothyroxine"],
    "shelcal": ["calcium carbonate", "vitamin d3"], "becosules": ["vitamin b complex"],
    "limcee": ["vitamin c"],
}

GENERIC_CLASS = {
    "paracetamol": "analgesic/antipyretic", "ibuprofen": "NSAID", "diclofenac": "NSAID",
    "aceclofenac": "NSAID", "aspirin": "antiplatelet/NSAID", "amoxicillin": "antibiotic",
    "clavulanic acid": "beta-lactamase inhibitor", "azithromycin": "macrolide antibiotic",
    "ciprofloxacin": "fluoroquinolone antibiotic", "cefixime": "cephalosporin antibiotic",
    "metronidazole": "antibiotic/antiprotozoal", "clarithromycin": "macrolide antibiotic",
    "pantoprazole": "PPI (acidity)", "omeprazole": "PPI (acidity)", "rabeprazole": "PPI (acidity)",
    "antacid (aluminium/magnesium hydroxide)": "antacid", "ondansetron": "anti-emetic",
    "domperidone": "anti-emetic/prokinetic", "fexofenadine": "antihistamine",
    "cetirizine": "antihistamine", "levocetirizine": "antihistamine",
    "montelukast": "anti-asthma (leukotriene)", "salbutamol": "bronchodilator",
    "etofylline": "bronchodilator", "theophylline": "bronchodilator",
    "prednisolone": "corticosteroid", "metformin": "antidiabetic", "glimepiride": "antidiabetic",
    "sitagliptin": "antidiabetic", "telmisartan": "antihypertensive (ARB)",
    "losartan": "antihypertensive (ARB)", "amlodipine": "antihypertensive (CCB)",
    "clopidogrel": "antiplatelet", "atorvastatin": "statin", "rosuvastatin": "statin",
    "furosemide": "diuretic", "spironolactone": "diuretic (K-sparing)", "digoxin": "cardiac glycoside",
    "warfarin": "anticoagulant", "levothyroxine": "thyroid hormone",
    "calcium carbonate": "supplement", "vitamin d3": "supplement",
    "vitamin b complex": "supplement", "vitamin c": "supplement",
}

# generic names usable directly as tokens (single-word ones)
for _g in list(GENERIC_CLASS):
    if " " not in _g and "(" not in _g:
        BRANDS.setdefault(_g, [_g])

CANONICAL = sorted(BRANDS)  # all valid drug tokens

NSAIDS = {"ibuprofen", "diclofenac", "aceclofenac"}
ARBS = {"telmisartan", "losartan"}
QT_DRUGS = {"azithromycin", "clarithromycin", "ondansetron", "domperidone", "ciprofloxacin"}
CHELATORS = {"calcium carbonate", "antacid (aluminium/magnesium hydroxide)"}

# (set_a, set_b, severity, message)
PAIR_RULES = [
    ({"warfarin"}, NSAIDS | {"aspirin", "clopidogrel"}, "MAJOR", "Bleeding risk increases sharply when combined with an anticoagulant."),
    ({"warfarin"}, {"metronidazole"}, "MAJOR", "Metronidazole strongly raises warfarin levels (INR) - bleeding risk."),
    ({"warfarin"}, {"ciprofloxacin", "azithromycin", "clarithromycin"}, "MODERATE", "Antibiotic may raise INR; monitor."),
    ({"clopidogrel"}, {"omeprazole"}, "MODERATE", "Omeprazole reduces clopidogrel activation; pantoprazole is usually preferred."),
    ({"aspirin"}, NSAIDS, "MODERATE", "Extra GI-bleeding risk; ibuprofen can blunt aspirin's heart protection."),
    ({"prednisolone"}, NSAIDS | {"aspirin"}, "MODERATE", "Steroid + NSAID raises stomach ulcer/bleeding risk."),
    ({"spironolactone"}, ARBS, "MAJOR", "High potassium (hyperkalemia) risk; check K+ levels."),
    ({"digoxin"}, {"furosemide"}, "MODERATE", "Diuretic can lower potassium and trigger digoxin toxicity."),
    ({"digoxin"}, {"clarithromycin"}, "MAJOR", "Clarithromycin raises digoxin levels."),
    ({"atorvastatin"}, {"clarithromycin"}, "MAJOR", "Raised statin levels - muscle damage (myopathy) risk."),
    ({"levothyroxine"}, CHELATORS, "MODERATE", "Reduces thyroid tablet absorption - keep a 4 hour gap."),
    ({"ciprofloxacin"}, CHELATORS, "MODERATE", "Reduces ciprofloxacin absorption - take cipro 2 h before / 6 h after."),
    ({"glimepiride"}, {"ciprofloxacin"}, "MODERATE", "Fluoroquinolones can swing blood sugar (hypo/hyper)."),
    (NSAIDS, ARBS, "MODERATE", "NSAID may reduce BP control and stress kidneys."),
]


def check_interactions(generics):
    """generics: list of (source_label, generic). Returns list of dict alerts."""
    alerts, seen = [], set()
    names = [g for _, g in generics]
    uniq = set(names)
    # duplicate ingredient from different products (e.g. Crocin + Combiflam)
    for g in uniq:
        srcs = sorted({s for s, x in generics if x == g})
        if len(srcs) > 1:
            alerts.append({"severity": "MAJOR", "drugs": " + ".join(srcs),
                           "issue": f"Duplicate ingredient '{g}' - overdose risk."})
    for a, b, sev, msg in PAIR_RULES:
        for x in uniq & a:
            for y in uniq & b:
                if x == y:
                    continue
                key = tuple(sorted((x, y))) + (msg,)
                if key in seen:
                    continue
                seen.add(key)
                alerts.append({"severity": sev, "drugs": f"{x} + {y}", "issue": msg})
    nsaids = sorted(uniq & NSAIDS)
    if len(nsaids) > 1:
        alerts.append({"severity": "MAJOR", "drugs": " + ".join(nsaids),
                       "issue": "Two NSAIDs together - stomach bleeding & kidney risk."})
    if uniq & NSAIDS and uniq & ARBS and "furosemide" in uniq:
        alerts.append({"severity": "MAJOR", "drugs": "NSAID + ARB + furosemide",
                       "issue": "'Triple whammy' - acute kidney injury risk."})
    qt = sorted(uniq & QT_DRUGS)
    if len(qt) > 1:
        alerts.append({"severity": "MODERATE", "drugs": " + ".join(qt),
                       "issue": "Combined QT-prolongation risk (heart rhythm)."})
    order = {"MAJOR": 0, "MODERATE": 1}
    return sorted(alerts, key=lambda d: order[d["severity"]])


# ---------- Rx shorthand ----------
FREQ = {
    "od": (1, "once a day", "दिन में एक बार"), "qd": (1, "once a day", "दिन में एक बार"),
    "bd": (2, "twice a day", "दिन में दो बार"), "bid": (2, "twice a day", "दिन में दो बार"),
    "tds": (3, "three times a day", "दिन में तीन बार"), "tid": (3, "three times a day", "दिन में तीन बार"),
    "qid": (4, "four times a day", "दिन में चार बार"), "hs": (1, "at bedtime", "सोने से पहले"),
    "sos": (0, "only when needed", "ज़रूरत पड़ने पर"), "prn": (0, "only when needed", "ज़रूरत पड़ने पर"),
    "stat": (1, "immediately, once", "तुरंत, एक बार"),
}
INSTR = {
    "ac": ("before food", "खाने से पहले"), "pc": ("after food", "खाने के बाद"),
    "before": ("before food", "खाने से पहले"), "after": ("after food", "खाने के बाद"),
    "empty": ("on an empty stomach", "खाली पेट"),
}
FORMS = {"tab": ("tablet", "गोली"), "tabs": ("tablet", "गोली"), "cap": ("capsule", "कैप्सूल"),
         "syp": ("syrup", "सिरप"), "syr": ("syrup", "सिरप"), "inj": ("injection", "इंजेक्शन"),
         "drops": ("drops", "ड्रॉप्स"), "susp": ("suspension", "सस्पेंशन")}
SLOTS_EN = ["morning", "afternoon", "night"]
SLOTS_HI = ["सुबह", "दोपहर", "रात"]


In [ ]:
%%writefile mediosis/rx_correct.py
"""fastText subword drug-name corrector.

Idea: OCR breaks drug names ("Azithrornycin"). fastText builds a word vector from its
character n-grams, so a misspelt word still lands near the real one. We train fastText on
(a) synthetic prescription lines and (b) 'alias sentences' where each canonical drug
co-occurs with simulated OCR corruptions of itself, then rank candidates by an ensemble of
fastText cosine similarity and edit-distance similarity.
"""
import random

import numpy as np
from gensim.models import FastText
from rapidfuzz import fuzz

from .lexicon import CANONICAL
from .rx_data import ocr_noise, synth_dataset

DIM = 100


def build_corpus(seed=0, n_lines=20000, n_alias=40):
    rng = random.Random(seed)
    corpus = [toks for toks, _ in synth_dataset(n_lines, seed=seed)]
    for d in CANONICAL:
        for _ in range(n_alias):
            sent = [d] + [ocr_noise(d, rng=rng) for _ in range(4)] + [d]
            rng.shuffle(sent)
            corpus.append(sent)
    return corpus


def train_fasttext(seed=0, epochs=10):
    corpus = build_corpus(seed)
    ft = FastText(vector_size=DIM, window=4, min_count=1, sg=1, min_n=2, max_n=5,
                  workers=4, seed=seed, bucket=200000)
    ft.build_vocab(corpus_iterable=corpus)
    ft.train(corpus_iterable=corpus, total_examples=len(corpus), epochs=epochs)
    return ft


class DrugCorrector:
    def __init__(self, ft, alpha=0.5, threshold=0.55):
        self.ft, self.alpha, self.threshold = ft, alpha, threshold
        M = np.stack([ft.wv.get_vector(c) for c in CANONICAL])
        self.M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

    def scores(self, word, mode="ensemble"):
        word = word.lower()
        v = self.ft.wv.get_vector(word)  # works for OOV via n-grams
        cos = self.M @ (v / (np.linalg.norm(v) + 1e-9))
        lev = np.array([fuzz.ratio(word, c) / 100 for c in CANONICAL])
        if mode == "fasttext":
            return cos
        if mode == "fuzzy":
            return lev
        return self.alpha * cos + (1 - self.alpha) * lev

    def correct(self, word, mode="ensemble"):
        s = self.scores(word, mode)
        i = int(np.argmax(s))
        conf = float(s[i])
        return (CANONICAL[i] if conf >= self.threshold else None), conf


def evaluate(corrector, n_per_drug=20, seed=123, n_edits=None, tests=None):
    """Top-1 accuracy on held-out OCR corruptions (different seed than training)."""
    rng = random.Random(seed)
    tests = tests or [(ocr_noise(d, n_edits=n_edits, rng=rng), d) for d in CANONICAL for _ in range(n_per_drug)]
    out = {}
    for mode in ["fuzzy", "fasttext", "ensemble"]:
        hit = sum(CANONICAL[int(np.argmax(corrector.scores(w, mode)))] == d for w, d in tests)
        out[mode] = hit / len(tests)
    return out, len(tests)


In [ ]:
%%writefile mediosis/rx_data.py
"""Tokenizer, OCR-noise simulator and synthetic prescription generator (BIO-tagged)."""
import random
import re

from .lexicon import CANONICAL

TOKEN_RE = re.compile(
    r"\d+(?:\.\d+)?(?:mg|ml|mcg|gm|g|iu)\b"   # 650mg
    r"|\d-\d-\d"                                # 1-0-1
    r"|[a-z0-9]*[a-z][a-z0-9]*"                 # words (OCR may put digits inside: 0mez)
    r"|\d+(?:\.\d+)?"                           # numbers
)


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


# Typical OCR confusions (both directions)
_CONF = [("o", "0"), ("l", "1"), ("i", "l"), ("m", "rn"), ("c", "e"), ("a", "o"),
         ("s", "5"), ("b", "6"), ("g", "9"), ("z", "2"), ("d", "cl"), ("n", "h"),
         ("u", "v"), ("e", "c"), ("t", "f"), ("r", "n")]


def ocr_noise(word, n_edits=None, rng=random):
    """Corrupt a word the way OCR / sloppy handwriting does."""
    w = word
    n_edits = n_edits or rng.choice([1, 1, 2])
    for _ in range(n_edits):
        op = rng.random()
        if op < 0.5:  # confusion substitution
            pairs = [(a, b) for a, b in _CONF if a in w] + [(b, a) for a, b in _CONF if b in w]
            if pairs:
                a, b = rng.choice(pairs)
                idx = [m.start() for m in re.finditer(re.escape(a), w)]
                i = rng.choice(idx)
                w = w[:i] + b + w[i + len(a):]
                continue
        if len(w) > 3 and op < 0.7:  # deletion
            i = rng.randrange(1, len(w))
            w = w[:i] + w[i + 1:]
        elif op < 0.85:  # insertion
            i = rng.randrange(len(w) + 1)
            w = w[:i] + rng.choice("aeilmnorst") + w[i:]
        elif len(w) > 2:  # adjacent swap
            i = rng.randrange(len(w) - 1)
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
    return w if w != word else word + rng.choice("aeil")


FORMS = ["tab", "cap", "syp", "inj", "tabs", "susp"]
STRENGTHS = ["250mg", "500mg", "650mg", "40mg", "20mg", "10mg", "5mg", "1g", "100mg", "200mg", "75mg", "5ml", "10ml"]
STRENGTHS_SPLIT = [("500", "mg"), ("650", "mg"), ("40", "mg"), ("20", "mg"), ("5", "ml")]
FREQS = [[f] for f in ["od", "bd", "bid", "tds", "tid", "qid", "hs", "sos", "stat",
                        "1-0-1", "1-1-1", "0-0-1", "1-0-0", "0-1-0", "1-1-0"]] + [
    ["once", "daily"], ["twice", "daily"], ["thrice", "daily"], ["daily"], ["at", "night"],
    ["three", "times", "a", "day"], ["twice", "a", "day"], ["when", "required"]]
INSTRS = [["after", "food"], ["before", "food"], ["ac"], ["pc"], ["empty", "stomach"], ["with", "milk"],
          ["after", "meals"], ["before", "breakfast"], ["with", "water"]]
DURS = [["x", "{n}", "days"], ["for", "{n}", "days"], ["{n}", "days"], ["x", "{n}", "d"],
        ["for", "{n}", "weeks"], ["x", "{n}", "wk"]]
FILLERS = ["rx", "dr", "advice", "take"] + [str(i) for i in range(1, 16)]
TRAILERS = [["if", "fever"], ["then", "stop"], ["do", "not", "skip"], ["review", "after"]]


def synth_line(rng=random, noise_p=0.35):
    """Return (tokens, tags) for one synthetic prescription line."""
    toks, tags = [], []

    def add(ts, label):
        for i, t in enumerate(ts):
            toks.append(t)
            tags.append(label if label == "O" else ("B-" if i == 0 else "I-") + label)

    if rng.random() < 0.4:
        add([rng.choice(FILLERS)], "O")
    if rng.random() < 0.8:
        add([rng.choice(FORMS)], "FORM")
    drug = rng.choice(CANONICAL)
    if rng.random() < noise_p:
        drug = ocr_noise(drug, rng=rng)
    add([drug], "DRUG")
    r = rng.random()
    if r < 0.6:
        add([rng.choice(STRENGTHS)], "STRENGTH")
    elif r < 0.8:
        add(list(rng.choice(STRENGTHS_SPLIT)), "STRENGTH")
    parts = []
    parts.append(("FREQ", list(rng.choice(FREQS))))
    if rng.random() < 0.6:
        parts.append(("INSTR", list(rng.choice(INSTRS))))
    if rng.random() < 0.75:
        n = str(rng.choice([3, 5, 7, 10, 14, 1, 2, 30]))
        parts.append(("DUR", [t.replace("{n}", n) for t in rng.choice(DURS)]))
    rng.shuffle(parts)
    for lab, ts in parts:
        add(ts, lab)
    if rng.random() < 0.15:
        add(rng.choice(TRAILERS), "O")
    return toks, tags


def synth_dataset(n, seed=0, noise_p=0.35):
    rng = random.Random(seed)
    return [synth_line(rng, noise_p) for _ in range(n)]


In [ ]:
%%writefile mediosis/rx_ner.py
"""BiLSTM-CRF tagger for prescription lines: DRUG / STRENGTH / FORM / FREQ / INSTR / DUR.

Input features per token = fastText subword vector (OOV-safe, so garbled OCR words still get a
meaningful vector) + 4 shape features. Spatial dropout is applied over the feature channels.
"""
import re

import numpy as np
import torch
import torch.nn as nn
from seqeval.metrics import classification_report, f1_score
from torchcrf import CRF

from .rx_data import synth_dataset

LABELS = ["O"] + [f"{p}-{t}" for t in ["DRUG", "STRENGTH", "FORM", "FREQ", "INSTR", "DUR"] for p in "BI"]
L2I = {l: i for i, l in enumerate(LABELS)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def shape(tok):
    return [float(tok[0].isdigit()),
            float(bool(re.search(r"(mg|ml|mcg|g|iu)$", tok)) and tok[0].isdigit()),
            float(bool(re.fullmatch(r"\d-\d-\d", tok))),
            min(len(tok), 15) / 15]


def featurize(ft, toks):
    return np.stack([np.concatenate([ft.wv.get_vector(t), shape(t)]) for t in toks]).astype(np.float32)


class BiLSTMCRF(nn.Module):
    def __init__(self, in_dim, hidden=128, n_tags=len(LABELS), dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.spatial_dropout = nn.Dropout1d(dropout)  # drops whole feature channels (SpatialDropout1D)
        self.lstm = nn.LSTM(hidden, hidden, num_layers=2, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.out = nn.Linear(2 * hidden, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def emissions(self, x):
        h = torch.relu(self.proj(x))                       # (B,T,H)
        h = self.spatial_dropout(h.transpose(1, 2)).transpose(1, 2)
        h, _ = self.lstm(h)
        return self.out(h)

    def loss(self, x, y, mask):
        return -self.crf(self.emissions(x), y, mask=mask, reduction="mean")

    def decode(self, x, mask):
        return self.crf.decode(self.emissions(x), mask=mask)


def _batch(ft, data):
    T = max(len(t) for t, _ in data)
    D = 100 + 4
    X = np.zeros((len(data), T, D), np.float32)
    Y = np.zeros((len(data), T), np.int64)
    M = np.zeros((len(data), T), bool)
    for i, (toks, tags) in enumerate(data):
        X[i, :len(toks)] = featurize(ft, toks)
        if tags:
            Y[i, :len(tags)] = [L2I[t] for t in tags]
        M[i, :len(toks)] = True
    return (torch.tensor(X, device=DEVICE), torch.tensor(Y, device=DEVICE),
            torch.tensor(M, device=DEVICE))


def train_ner(ft, n_train=8000, epochs=6, bs=64, seed=0, log=print):
    torch.manual_seed(seed)
    data = synth_dataset(n_train, seed=seed)
    model = BiLSTMCRF(104).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    for ep in range(epochs):
        model.train()
        perm = np.random.RandomState(ep).permutation(len(data))
        tot = 0
        for b in range(0, len(data), bs):
            X, Y, M = _batch(ft, [data[i] for i in perm[b:b + bs]])
            loss = model.loss(X, Y, M)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += loss.item()
        log(f"  NER epoch {ep + 1}/{epochs}  loss={tot / (len(data) / bs):.3f}")
    return model


@torch.no_grad()
def predict(model, ft, token_lists):
    model.eval()
    X, _, M = _batch(ft, [(t, None) for t in token_lists])
    return [[LABELS[i] for i in seq] for seq in model.decode(X, M)]


def evaluate_ner(model, ft, n=2000, seed=999, noise_p=0.6):
    """Held-out set: different seed + heavier OCR noise than training."""
    data = synth_dataset(n, seed=seed, noise_p=noise_p)
    pred = []
    for b in range(0, n, 256):
        pred += predict(model, ft, [t for t, _ in data[b:b + 256]])
    gold = [g for _, g in data]
    return f1_score(gold, pred), classification_report(gold, pred, digits=3)


def save(model, path):
    torch.save(model.state_dict(), path)


def load(path):
    m = BiLSTMCRF(104).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    return m


In [ ]:
%%writefile mediosis/rx_ocr.py
"""OCR for prescriptions: EasyOCR (detection + printed text) with optional TrOCR re-read for handwriting."""
import numpy as np
import torch
from PIL import Image

_reader, _trocr = None, None


def _easy():
    global _reader
    if _reader is None:
        import easyocr
        _reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)
    return _reader


def _tr():
    global _trocr
    if _trocr is None:
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        name = "microsoft/trocr-base-handwritten"
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        _trocr = (TrOCRProcessor.from_pretrained(name),
                  VisionEncoderDecoderModel.from_pretrained(name).to(dev), dev)
    return _trocr


def _group_lines(results):
    """results: [(box, text, conf)] -> list of lines [(text, conf, (x0,y0,x1,y1))] in reading order."""
    items = []
    for box, text, conf in results:
        xs, ys = [p[0] for p in box], [p[1] for p in box]
        items.append(dict(x0=min(xs), x1=max(xs), y0=min(ys), y1=max(ys), cy=(min(ys) + max(ys)) / 2,
                          h=max(ys) - min(ys), text=text, conf=conf))
    if not items:
        return []
    items.sort(key=lambda d: d["cy"])
    med_h = float(np.median([d["h"] for d in items]))
    lines, cur = [], [items[0]]
    for d in items[1:]:
        if abs(d["cy"] - np.mean([c["cy"] for c in cur])) < 0.6 * med_h:
            cur.append(d)
        else:
            lines.append(cur)
            cur = [d]
    lines.append(cur)
    out = []
    for ln in lines:
        ln.sort(key=lambda d: d["x0"])
        out.append((" ".join(d["text"] for d in ln), float(np.mean([d["conf"] for d in ln])),
                    (min(d["x0"] for d in ln), min(d["y0"] for d in ln),
                     max(d["x1"] for d in ln), max(d["y1"] for d in ln))))
    return out


def read_prescription(img, handwriting=False):
    """img: PIL.Image or np.ndarray. Returns list of (line_text, confidence)."""
    if isinstance(img, Image.Image):
        img = np.array(img.convert("RGB"))
    lines = _group_lines(_easy().readtext(img))
    if not handwriting:
        return [(t, c) for t, c, _ in lines]
    proc, model, dev = _tr()
    out = []
    for t, c, (x0, y0, x1, y1) in lines:
        pad = 4
        crop = Image.fromarray(img[max(0, int(y0) - pad):int(y1) + pad, max(0, int(x0) - pad):int(x1) + pad])
        with torch.no_grad():
            ids = model.generate(proc(images=crop, return_tensors="pt").pixel_values.to(dev), max_new_tokens=48)
        out.append((proc.batch_decode(ids, skip_special_tokens=True)[0], c))
    return out


In [ ]:
%%writefile mediosis/rx_pipeline.py
"""RxDecode pipeline: text lines -> NER -> drug correction -> generics -> interactions -> explanation."""
import re

from .lexicon import BRANDS, FORMS, FREQ, GENERIC_CLASS, INSTR, SLOTS_EN, SLOTS_HI, check_interactions
from .rx_data import tokenize
from .rx_ner import predict

PHRASE_FREQ = [("three times", "tds"), ("thrice", "tds"), ("twice", "bd"), ("four times", "qid"),
               ("once", "od"), ("at night", "hs"), ("when required", "sos"), ("daily", "od")]
UNIT_HI = {"day": "दिन", "week": "हफ़्ते", "month": "महीने"}


def _spans(toks, tags):
    ents, cur = [], None
    for t, g in zip(toks, tags):
        if g.startswith("B-") or (g.startswith("I-") and (cur is None or cur[0] != g[2:])):
            cur = [g[2:], [t]]
            ents.append(cur)
        elif g.startswith("I-"):
            cur[1].append(t)
        else:
            cur = None
    return [(lab, " ".join(ts)) for lab, ts in ents]


def _freq(text):
    if not text:
        return None, None, None
    m = re.search(r"(\d)-(\d)-(\d)", text)
    if m:
        slots = [int(x) for x in m.groups()]
        en = ", ".join(f"{n} in the {s}" for n, s in zip(slots, SLOTS_EN) if n)
        hi = ", ".join(f"{s} {n}" for n, s in zip(slots, SLOTS_HI) if n)
        return sum(slots), en, hi
    for tok in text.split():
        if tok in FREQ:
            return FREQ[tok]
    for ph, code in PHRASE_FREQ:
        if ph in text:
            return FREQ[code]
    return None, text, text


def _dur(text):
    if not text:
        return None, None
    m = re.search(r"(\d+)\s*(d|day|days|w|wk|week|weeks|m|month|months)?", text)
    if not m:
        return None, None
    n, u = m.group(1), (m.group(2) or "d")[0]
    unit = {"d": "day", "w": "week", "m": "month"}[u]
    return f"{n} {unit}{'s' if n != '1' else ''}", f"{n} {UNIT_HI[unit]}"


def _instr(text):
    if not text:
        return None, None
    for k, v in INSTR.items():
        if k in text.split():
            return v
    return text, text


_CONFUSE = str.maketrans({"O": "0", "o": "0", "S": "5", "s": "5", "I": "1", "l": "1", "B": "8"})


def clean_line(line):
    """OCR clean-up before tagging: drop list numbering, repair digit/letter swaps in strengths (4Omg -> 40mg)."""
    line = re.sub(r"^\s*\d{1,2}\s*[._)\-:,]+\s*", "", line)
    line = re.sub(r"^\s*\d{1,2}\s+(?=(tab|cap|syp|syr|inj|susp)\b)", "", line, flags=re.I)
    return re.sub(r"\b([0-9OoSsIlB]{1,4})(mg|ml|mcg)\b",
                  lambda m: m.group(1).translate(_CONFUSE) + m.group(2), line, flags=re.I)


def _has_rx_context(d):
    strength_ok = bool(re.search(r"\d", d.get("STRENGTH", "")))
    return "FORM" in d or _freq(d.get("FREQ"))[0] is not None or strength_ok


def decode_lines(lines, ner, ft, corrector):
    """lines: list of str. Returns (meds list, alerts list)."""
    lines = [clean_line(l) for l in lines]
    token_lists = [tokenize(l) for l in lines]
    keep = [i for i, t in enumerate(token_lists) if t]
    tags = predict(ner, ft, [token_lists[i] for i in keep]) if keep else []
    meds = []
    for i, tg in zip(keep, tags):
        ents = _spans(token_lists[i], tg)
        d = {}
        for lab, txt in ents:
            d.setdefault(lab, txt)
        if "DRUG" not in d:
            continue
        raw = d["DRUG"].split()[0]
        canon, conf = corrector.correct(raw)
        if canon is None or conf < 0.75:  # NER may have picked the wrong word: scan the line
            best = max(((corrector.correct(t), t) for t in token_lists[i] if t.isalpha() and len(t) >= 3),
                       key=lambda z: z[0][1], default=((None, 0.0), raw))
            if best[0][0] and best[0][1] >= 0.85:
                (canon, conf), raw = best
        ctx = _has_rx_context(d)
        if (canon is None or conf < 0.75) and not ctx:
            continue  # header / signature / address line, not a medicine
        generics = BRANDS.get(canon, []) if canon else []
        _, f_en, f_hi = _freq(d.get("FREQ"))
        du_en, du_hi = _dur(d.get("DUR"))
        in_en, in_hi = _instr(d.get("INSTR"))
        form_en, form_hi = FORMS.get((d.get("FORM") or "").split()[0] if d.get("FORM") else "", ("", ""))
        meds.append(dict(line=lines[i], ocr_word=raw, drug=canon or "UNRECOGNISED", match_conf=round(conf, 2),
                         generic=", ".join(generics) or "-",
                         drug_class=", ".join(GENERIC_CLASS.get(g, "") for g in generics) or "-",
                         form=form_en, form_hi=form_hi, strength=d.get("STRENGTH", ""),
                         freq_en=f_en, freq_hi=f_hi, dur_en=du_en, dur_hi=du_hi, instr_en=in_en, instr_hi=in_hi))
    pairs = [(m["drug"], g) for m in meds if m["drug"] != "UNRECOGNISED" for g in m["generic"].split(", ")]
    return meds, check_interactions(pairs)


def explain(meds, lang="en"):
    out = []
    for i, m in enumerate(meds, 1):
        name = m["drug"].title()
        if lang == "hi":
            parts = [p for p in [m["freq_hi"], m["instr_hi"], (m["dur_hi"] + " तक") if m["dur_hi"] else None] if p]
            out.append(f"{i}. **{name}** ({m['generic']}) {m['form_hi']} {m['strength']} — " + ", ".join(parts) + "।")
        else:
            parts = [p for p in [m["freq_en"], m["instr_en"], ("for " + m["dur_en"]) if m["dur_en"] else None] if p]
            out.append(f"{i}. **{name}** ({m['generic']}; {m['drug_class']}) {m['form']} {m['strength']} — "
                       + ", ".join(parts) + ".")
        if m["drug"] == "UNRECOGNISED":
            out[-1] += "  ⚠️ could not confidently read this drug name - check with your pharmacist."
        elif m["match_conf"] < 0.75:
            out[-1] += f"  ⚠️ low read confidence ({m['match_conf']}) - verify the name."
    return "\n".join(out)


In [ ]:
%%writefile mediosis/rx_realtest.py
"""Hand-written, real-style prescription lines (formats NOT produced by the generator templates,
plus realistic OCR errors). Used as an honest out-of-template test for the NER tagger."""
from .rx_data import tokenize

# each line = list of (text chunk, label)  ; label "O" = outside
REAL_LINES = [
    [("Tab.", "FORM"), ("Dolo", "DRUG"), ("650", "STRENGTH"), ("1 tab", "O"), ("thrice daily", "FREQ"), ("for 3 days", "DUR")],
    [("Rx", "O"), ("Augmentin", "DRUG"), ("625 mg", "STRENGTH"), ("BD", "FREQ"), ("after meals", "INSTR"), ("x 5 days", "DUR")],
    [("Cap", "FORM"), ("Pantocid", "DRUG"), ("40mg", "STRENGTH"), ("OD", "FREQ"), ("before breakfast", "INSTR")],
    [("Tab", "FORM"), ("Azithrornycin", "DRUG"), ("500 mg", "STRENGTH"), ("once daily", "FREQ"), ("x 3 days", "DUR")],
    [("Syp", "FORM"), ("Calpo1", "DRUG"), ("5 ml", "STRENGTH"), ("SOS", "FREQ")],
    [("Tab", "FORM"), ("Glycornet", "DRUG"), ("500mg", "STRENGTH"), ("1-0-1", "FREQ"), ("after food", "INSTR"), ("for 30 days", "DUR")],
    [("Tab", "FORM"), ("Telma", "DRUG"), ("40", "STRENGTH"), ("0-0-1", "FREQ"), ("continue", "O")],
    [("Tab", "FORM"), ("Ecosprin", "DRUG"), ("75mg", "STRENGTH"), ("HS", "FREQ"), ("x 1 month", "DUR")],
    [("Tab", "FORM"), ("Combiflarn", "DRUG"), ("SOS", "FREQ"), ("for pain", "O")],
    [("Cap", "FORM"), ("0mez", "DRUG"), ("20", "STRENGTH"), ("BD", "FREQ"), ("AC", "INSTR"), ("x 14 days", "DUR")],
    [("Tab", "FORM"), ("Montair", "DRUG"), ("10 mg", "STRENGTH"), ("at night", "FREQ"), ("for 2 weeks", "DUR")],
    [("Tab", "FORM"), ("Thyronorm", "DRUG"), ("50mcg", "STRENGTH"), ("1-0-0", "FREQ"), ("empty stomach", "INSTR")],
    [("Inj", "FORM"), ("Ondem", "DRUG"), ("4mg", "STRENGTH"), ("STAT", "FREQ")],
    [("Tab", "FORM"), ("Shelca1", "DRUG"), ("500", "STRENGTH"), ("OD", "FREQ"), ("after lunch", "INSTR"), ("x 1 month", "DUR")],
    [("Tab", "FORM"), ("Allegra", "DRUG"), ("120mg", "STRENGTH"), ("once daily", "FREQ"), ("x 5 d", "DUR")],
    [("Tab", "FORM"), ("Warf", "DRUG"), ("5", "STRENGTH"), ("OD", "FREQ"), ("as advised", "O")],
    [("Tab", "FORM"), ("Metrogy1", "DRUG"), ("400mg", "STRENGTH"), ("TDS", "FREQ"), ("x 5 days", "DUR")],
    [("Tab", "FORM"), ("Clopilet", "DRUG"), ("75", "STRENGTH"), ("0-1-0", "FREQ"), ("after lunch", "INSTR")],
]


def real_test_set():
    data = []
    for chunks in REAL_LINES:
        toks, tags = [], []
        for text, lab in chunks:
            ts = tokenize(text)
            toks += ts
            tags += ["O"] * len(ts) if lab == "O" else [("B-" if i == 0 else "I-") + lab for i in range(len(ts))]
        data.append((toks, tags))
    return data


# gold canonical names for the misspelt drugs above (for corrector eval)
REAL_DRUG_GOLD = {"dolo": "dolo", "augmentin": "augmentin", "pantocid": "pantocid",
                  "azithrornycin": "azithromycin", "calpo1": "calpol", "glycornet": "glycomet",
                  "telma": "telma", "ecosprin": "ecosprin", "combiflarn": "combiflam", "0mez": "omez",
                  "montair": "montair", "thyronorm": "thyronorm", "ondem": "ondem", "shelca1": "shelcal",
                  "allegra": "allegra", "warf": "warf", "metrogy1": "metrogyl", "clopilet": "clopilet"}


In [ ]:
%%writefile mediosis/xr_check.py
"""XCheck: cross-check what the image model sees against what the radiology report says."""
from .xr_report import analyse_report

POS, NEG = 0.60, 0.40    # model is calibrated so 0.5 = operating point
UNSTABLE_STD = 0.08      # TTA disagreement => unstable prediction

# labels the model is weaker on / that are often not reported -> lower severity
SOFT = {"Infiltration", "Lung Opacity", "Pleural_Thickening", "Fibrosis", "Lung Lesion"}


def image_status(p, sd):
    if sd > UNSTABLE_STD:
        return "unstable"
    return "positive" if p >= POS else "negative" if p <= NEG else "borderline"


def cross_check(probs, report_text, asserter):
    """probs: {pathology: (mean, std)}. Returns (rows, report_findings)."""
    rep = analyse_report(report_text, asserter) if report_text and report_text.strip() else {}
    rows = []
    for lab, (p, sd) in sorted(probs.items(), key=lambda kv: -kv[1][0]):
        img = image_status(p, sd)
        r_status, evidence, engine = rep.get(lab, ("not mentioned", "", ""))
        flag, sev = "✅ consistent", 0
        if img == "positive" and r_status == "absent":
            flag, sev = "🔴 CONTRADICTION: image suggests it, report says absent", 3
        elif img == "positive" and r_status == "not mentioned":
            flag, sev = ("🟠 possible MISSED finding (not in report)", 2) if lab not in SOFT else ("🟡 unmentioned (low-specificity label)", 1)
        elif img == "negative" and r_status == "present":
            flag, sev = "🟠 report claims it, image model does not support it", 2
        elif img in ("borderline", "unstable") and r_status in ("present", "absent"):
            flag, sev = f"🟡 image {img} — radiologist judgement prevails", 1
        elif r_status == "uncertain" and img == "positive":
            flag, sev = "🟡 report uncertain, image leans positive — consider follow-up", 1
        elif img == "negative" and r_status == "not mentioned":
            continue  # nothing to say
        rows.append(dict(pathology=lab, image_prob=round(p, 3), tta_std=round(sd, 3), image=img,
                         report=r_status, evidence=evidence, engine=engine, flag=flag, severity=sev))
    rows.sort(key=lambda r: (-r["severity"], -r["image_prob"]))
    return rows, rep


In [ ]:
%%writefile mediosis/xr_report.py
"""Radiology-report NLP: find each pathology mention and decide if the report says it is
PRESENT, ABSENT or UNCERTAIN.

Two assertion engines:
  * NegEx-style rules (baseline)
  * Keras BiGRU + self-attention classifier (Embedding -> SpatialDropout1D -> BiGRU with
    recurrent_dropout -> MultiHeadAttention self-attention -> pooling -> softmax), with our own
    vocabulary hash table and PRE-padding.
"""
import json
import os
import random
import re

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
import numpy as np

CLASSES = ["present", "absent", "uncertain"]
MAXLEN = 40

SYN = {
    "Atelectasis": ["atelectasis", "atelectatic", "collapse"],
    "Consolidation": ["consolidation", "consolidations", "airspace disease"],
    "Infiltration": ["infiltrate", "infiltrates", "infiltration"],
    "Pneumothorax": ["pneumothorax", "pneumothoraces"],
    "Edema": ["pulmonary edema", "edema", "oedema", "vascular congestion"],
    "Emphysema": ["emphysema", "hyperinflation", "hyperinflated lungs"],
    "Fibrosis": ["fibrosis", "fibrotic changes", "scarring"],
    "Effusion": ["pleural effusions", "pleural effusion", "effusions", "effusion", "costophrenic blunting"],
    "Pneumonia": ["pneumonia", "pneumonic"],
    "Pleural_Thickening": ["pleural thickening"],
    "Cardiomegaly": ["cardiomegaly", "enlarged heart", "cardiac enlargement", "heart is enlarged",
                     "heart size is enlarged", "enlarged cardiac silhouette"],
    "Nodule": ["nodules", "nodule", "nodular opacity"],
    "Mass": ["mass", "masses"],
    "Lung Lesion": ["lesion", "lesions"],
    "Fracture": ["fractures", "fracture"],
    "Lung Opacity": ["opacities", "opacity", "opacification"],
    "Enlarged Cardiomediastinum": ["widened mediastinum", "mediastinal widening", "enlarged cardiomediastinal silhouette"],
    "Hernia": ["hiatal hernia", "hernia"],
}
PHRASES = sorted([(s, lab) for lab, ss in SYN.items() for s in ss], key=lambda x: -len(x[0]))

NORMAL_PATTERNS = [
    (r"heart size (is )?(normal|within normal limits)|normal heart size|heart is normal in size|"
     r"(cardiac|cardiomediastinal) silhouette (is )?(normal|within normal limits|unremarkable)|normal cardiac silhouette",
     ["Cardiomegaly", "Enlarged Cardiomediastinum"]),
    (r"lungs are clear|clear lungs|lungs are well expanded and clear",
     ["Consolidation", "Lung Opacity", "Pneumonia", "Edema", "Infiltration", "Atelectasis"]),
    (r"no acute cardiopulmonary (process|abnormality|disease|findings?)",
     ["Consolidation", "Lung Opacity", "Pneumonia", "Edema", "Effusion", "Pneumothorax", "Cardiomegaly"]),
]

PRE_NEG = ["no", "without", "negative for", "free of", "no evidence of", "absence of", "resolution of",
           "no signs of", "not", "nor", "clear of", "ruled out"]
POST_NEG = ["is not seen", "are not seen", "not identified", "is absent", "are absent", "has resolved",
            "have resolved", "is ruled out", "not present", "not demonstrated", "no longer visualized", "no longer seen"]
UNCERTAIN = ["possible", "possibly", "may represent", "may be", "cannot exclude", "can not exclude",
             "cannot be excluded", "not excluded", "not entirely excluded", "suspicious for", "questionable",
             "likely", "suggestive of", "concerning for", "probable", "could represent", "differential", "versus",
             "?", "borderline", "equivocal"]
SCOPE_BREAK = r"\b(but|however|although|though|with interval|which)\b"


def sentences(text):
    return [s.strip() for s in re.split(r"[.;\n]+", text.lower()) if s.strip()]


def mentions(sentence):
    """Find pathology mentions (longest phrase first, no overlaps). Returns [(label, start, end)]."""
    taken, out = [], []
    for ph, lab in PHRASES:
        for m in re.finditer(r"\b" + re.escape(ph) + r"\b", sentence):
            if any(m.start() < e and m.end() > s for s, e in taken):
                continue
            taken.append((m.start(), m.end()))
            out.append((lab, m.start(), m.end()))
    return sorted(out, key=lambda x: x[1])


def mask_sentence(sentence, target, all_mentions):
    """Replace target mention with 'tgt' and other findings with 'oth'."""
    s, offset = sentence, 0
    for lab, a, b in all_mentions:
        rep = " tgt " if (a, b) == target else " oth "
        s = s[:a + offset] + rep + s[b + offset:]
        offset += len(rep) - (b - a)
    return s


def words(s):
    return re.findall(r"[a-z]+|\?", s)


# ---------------- rule baseline (NegEx-style) ----------------
def rule_assert(masked):
    clause = [c for c in re.split(SCOPE_BREAK, masked) if "tgt" in c]
    c = " " + (clause[0] if clause else masked) + " "
    ws = words(c)
    if "tgt" not in ws:
        return "present"
    i = ws.index("tgt")
    pre = " " + " ".join(ws[max(0, i - 6):i]) + " "
    post = " " + " ".join(ws[i + 1:i + 7]) + " "
    if any(f" {u} " in f" {c} " or (u == "?" and "?" in c) for u in UNCERTAIN):
        return "uncertain"
    if any(f" {n} " in pre for n in PRE_NEG) or any(f" {n} " in post for n in POST_NEG):
        return "absent"
    return "present"


# ---------------- synthetic training data ----------------
TRAIN_T = {
    "present": ["there is {mod} {f}", "{f} is seen in the {loc}", "{f} noted", "findings consistent with {f}",
                "interval increase in {f}", "{loc} {f} is present", "{mod} {loc} {f}", "new {f} in the {loc}",
                "persistent {f}", "{f} is again demonstrated", "redemonstrated {f}", "{f} has developed"],
    "absent": ["no {f}", "no evidence of {f}", "there is no {f}", "negative for {f}", "{f} is not seen",
               "without {f}", "{f} has resolved", "free of {f}", "no {loc} {f}", "no signs of {f}",
               "resolution of previously seen {f}", "{f} not identified"],
    "uncertain": ["possible {f}", "cannot exclude {f}", "may represent {f}", "suspicious for {f}",
                  "questionable {loc} {f}", "{f} is not entirely excluded", "likely {f}", "concerning for {f}",
                  "probable {mod} {f}", "could represent {f}", "{loc} opacity ? {f}", "suggestive of {f}"],
}
TEST_T = {  # held-out wording, never seen in training
    "present": ["the study demonstrates {mod} {f}", "{f} is identified at the {loc}", "evidence of {f} is noted",
                "{mod} {f} remains"],
    "absent": ["there is no evidence of {f}", "no definite {f} is seen", "{f} are absent", "no new {f}"],
    "uncertain": ["{f} cannot be excluded", "findings may be due to {f}", "equivocal {f}", "differential includes {f}"],
}
MODS = ["small", "moderate", "large", "mild", "minimal", "trace", "subtle", "extensive", "patchy"]
LOCS = ["right base", "left base", "right upper lobe", "left lower lobe", "bilateral", "right", "left",
        "retrocardiac", "perihilar", "lingular", "apical"]
JOINERS = [" but ", ", ", " and ", " with ", ". however, "]


def _one(lab, templates, rng):
    f = rng.choice(PHRASES)[0]
    s = rng.choice(templates[lab]).format(f="TGT_" + f, mod=rng.choice(MODS), loc=rng.choice(LOCS))
    return s


FILL = ["the", "is", "are", "seen", "at", "in", "of", "a", "there", "again", "on", "study", "today", "this",
        "compared", "prior", "exam", "chest", "radiograph", "findings", "lungs", "lobe", "zone", "which",
        "stable", "unchanged", "within", "region", "view", "frontal", "lateral", "portable", "patient"]
PRESENT_CUES = ["is seen", "is noted", "demonstrates", "remains", "is identified", "is evident", "shows",
                "persists", "has increased", "has developed", "with", "and", "consistent with", "evidence of",
                "compatible with", "represents", "is present", "is again seen"]
POST_UNC = ["cannot be excluded", "not excluded", "not entirely excluded", "is questioned", "is possible",
            "is suspected", "is likely", "is not ruled out"]


def _lexicon_sentence(lab, rng):
    """Compose sentences straight from the trigger lexicons + random filler, so the network learns
    the triggers themselves rather than memorising whole templates."""
    f = "TGT_" + rng.choice(PHRASES)[0]
    fill = lambda k: " ".join(rng.choice(FILL) for _ in range(rng.randint(0, k)))
    mod = rng.choice(MODS + LOCS + [""])
    if lab == "absent":
        core = (f"{rng.choice(PRE_NEG)} {fill(2)} {mod} {f}" if rng.random() < 0.6
                else f"{mod} {f} {rng.choice(POST_NEG)}")
    elif lab == "uncertain":
        u = [x for x in UNCERTAIN if x != "?"]
        core = (f"{rng.choice(u)} {fill(2)} {mod} {f}" if rng.random() < 0.6
                else f"{mod} {f} {rng.choice(POST_UNC)}")
    else:
        core = (f"{mod} {f} {rng.choice(PRESENT_CUES)} {fill(2)}" if rng.random() < 0.5
                else f"{fill(2)} {rng.choice(PRESENT_CUES)} {mod} {f}")
    return re.sub(r"\s+", " ", f"{fill(3)} {core} {fill(2)}").strip()


def synth_assertions(n, templates, seed, lexicon_p=0.0):
    rng = random.Random(seed)
    data = []
    for _ in range(n):
        lab = rng.choice(CLASSES)
        s = _lexicon_sentence(lab, rng) if rng.random() < lexicon_p else _one(lab, templates, rng)
        if rng.random() < 0.45:  # add a distractor clause about another finding
            other = rng.choice(CLASSES)
            o = (_lexicon_sentence(other, rng) if rng.random() < lexicon_p
                 else _one(other, templates, rng)).replace("TGT_", "")
            s = (s + rng.choice(JOINERS) + o) if rng.random() < 0.5 else (o + rng.choice(JOINERS) + s)
        s = s.lower()
        # mask: target -> tgt, any other finding -> oth
        s_clean = s.replace("tgt_", "@@")
        ms = mentions(s_clean.replace("@@", ""))
        # find target position = mention whose start matches the @@ marker
        pos = s_clean.index("@@")
        plain = s_clean.replace("@@", "")
        target = next(((a, b) for _, a, b in ms if a == pos), None)
        if target is None:
            continue
        data.append((mask_sentence(plain, target, ms), lab))
    return data


# hand-written realistic report sentences (sentence, pathology, gold)
REAL_SET = [
    ("No pleural effusion or pneumothorax.", "Effusion", "absent"),
    ("No pleural effusion or pneumothorax.", "Pneumothorax", "absent"),
    ("Mild cardiomegaly is again noted.", "Cardiomegaly", "present"),
    ("Patchy opacity at the left base may represent atelectasis or pneumonia.", "Pneumonia", "uncertain"),
    ("Patchy opacity at the left base may represent atelectasis or pneumonia.", "Atelectasis", "uncertain"),
    ("Small right pleural effusion has increased in size.", "Effusion", "present"),
    ("There is no focal consolidation.", "Consolidation", "absent"),
    ("Pulmonary edema has resolved.", "Edema", "absent"),
    ("Moderate pulmonary edema with bilateral effusions.", "Edema", "present"),
    ("Moderate pulmonary edema with bilateral effusions.", "Effusion", "present"),
    ("Cannot exclude a small left apical pneumothorax.", "Pneumothorax", "uncertain"),
    ("A 1 cm nodule is seen in the right upper lobe.", "Nodule", "present"),
    ("No displaced rib fracture is identified.", "Fracture", "absent"),
    ("Hyperinflated lungs consistent with emphysema.", "Emphysema", "present"),
    ("Right lower lobe consolidation concerning for pneumonia.", "Consolidation", "present"),
    ("Right lower lobe consolidation concerning for pneumonia.", "Pneumonia", "uncertain"),
    ("Small hiatal hernia.", "Hernia", "present"),
    ("Left basilar atelectasis without effusion.", "Atelectasis", "present"),
    ("Left basilar atelectasis without effusion.", "Effusion", "absent"),
    ("No new opacity to suggest pneumonia.", "Pneumonia", "absent"),
    ("No new opacity to suggest pneumonia.", "Lung Opacity", "absent"),
    ("Possible small right effusion.", "Effusion", "uncertain"),
    ("Biapical pleural thickening and scarring.", "Pleural_Thickening", "present"),
    ("Biapical pleural thickening and scarring.", "Fibrosis", "present"),
    ("There is no pneumothorax but a small effusion persists.", "Pneumothorax", "absent"),
    ("There is no pneumothorax but a small effusion persists.", "Effusion", "present"),
    ("The previously seen mass is no longer visualized.", "Mass", "absent"),
    ("Widened mediastinum, aortic injury not excluded.", "Enlarged Cardiomediastinum", "present"),
    ("Interstitial infiltrates are likely infectious.", "Infiltration", "present"),
    ("Retrocardiac opacity, atelectasis versus consolidation.", "Atelectasis", "uncertain"),
]


def real_examples():
    out = []
    for sent, lab, gold in REAL_SET:
        s = sentences(sent)[0]
        ms = mentions(s)
        tgt = next(((a, b) for l, a, b in ms if l == lab), None)
        if tgt:
            out.append((mask_sentence(s, tgt, ms), gold))
    return out


# ---------------- Keras model ----------------
class Vocab:
    """'Active vocabulary' hash table: word -> id (0 = PAD, 1 = OOV)."""

    def __init__(self, itos=None):
        self.itos = itos or ["<pad>", "<oov>"]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def fit(self, texts, min_freq=1):
        from collections import Counter
        c = Counter(w for t in texts for w in words(t))
        for w, n in c.most_common():
            if n >= min_freq and w not in self.stoi:
                self.stoi[w] = len(self.itos)
                self.itos.append(w)
        return self

    def encode(self, texts):
        from keras.utils import pad_sequences
        seqs = [[self.stoi.get(w, 1) for w in words(t)] for t in texts]
        return pad_sequences(seqs, maxlen=MAXLEN, padding="pre", truncating="pre")  # PRE-padding


def build_model(vocab_size):
    import keras
    from keras import layers
    inp = keras.Input((MAXLEN,), dtype="int32")
    x = layers.Embedding(vocab_size, 64)(inp)
    x = layers.SpatialDropout1D(0.2)(x)
    x = layers.Bidirectional(layers.GRU(64, return_sequences=True, recurrent_dropout=0.1))(x)
    att = layers.MultiHeadAttention(num_heads=2, key_dim=32, name="self_attention")(x, x)  # Q=K=V=x
    x = layers.LayerNormalization()(x + att)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(3, activation="softmax")(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(2e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def train_assertion(n=20000, epochs=6, seed=0, verbose=2):
    import keras
    keras.utils.set_random_seed(seed)
    train = synth_assertions(n, TRAIN_T, seed, lexicon_p=0.6)
    vocab = Vocab().fit([s for s, _ in train])
    X = vocab.encode([s for s, _ in train])
    y = np.array([CLASSES.index(l) for _, l in train])
    model = build_model(len(vocab.itos))
    model.fit(X, y, epochs=epochs, batch_size=128, validation_split=0.1, verbose=verbose)
    return model, vocab


class Asserter:
    def __init__(self, model, vocab, min_conf=0.7):
        self.model, self.vocab, self.min_conf = model, vocab, min_conf

    def predict(self, masked_list):
        if not masked_list:
            return []
        P = self.model.predict(self.vocab.encode(masked_list), verbose=0)
        out = []
        for s, p in zip(masked_list, P):
            k = int(p.argmax())
            if p[k] >= self.min_conf:
                out.append((CLASSES[k], float(p[k]), "model"))
            else:
                out.append((rule_assert(s), float(p[k]), "rules (low model conf)"))
        return out

    def save(self, d):
        os.makedirs(d, exist_ok=True)
        self.model.save(os.path.join(d, "assertion.keras"))
        json.dump(self.vocab.itos, open(os.path.join(d, "vocab.json"), "w"))

    @classmethod
    def load(cls, d):
        import keras
        return cls(keras.models.load_model(os.path.join(d, "assertion.keras")),
                   Vocab(json.load(open(os.path.join(d, "vocab.json")))))


def evaluate_assertion(asserter, seed=7):
    from sklearn.metrics import accuracy_score, f1_score
    res = {}
    sets = {"synthetic held-out templates": synth_assertions(1500, TEST_T, seed),
            "hand-written real-style (n=%d)" % len(REAL_SET): real_examples()}
    for name, data in sets.items():
        gold = [l for _, l in data]
        rules = [rule_assert(s) for s, _ in data]
        model = [p[0] for p in asserter.predict([s for s, _ in data])]
        res[name] = {"NegEx rules": (accuracy_score(gold, rules), f1_score(gold, rules, average="macro")),
                     "BiGRU+self-attn (+rule fallback)": (accuracy_score(gold, model), f1_score(gold, model, average="macro"))}
    return res


def analyse_report(text, asserter):
    """Returns {pathology: (status, evidence_sentence, engine)} for everything the report talks about."""
    found = {}
    rank = {"present": 0, "uncertain": 1, "absent": 2}
    jobs = []
    for s in sentences(text):
        ms = mentions(s)
        for lab, a, b in ms:
            jobs.append((lab, s, mask_sentence(s, (a, b), ms)))
    preds = asserter.predict([j[2] for j in jobs])
    for (lab, s, _), (status, conf, eng) in zip(jobs, preds):
        if lab not in found or rank[status] < rank[found[lab][0]]:
            found[lab] = (status, s, f"{eng} {conf:.2f}")
    low = text.lower()
    for pat, labs in NORMAL_PATTERNS:
        m = re.search(pat, low)
        if m:
            for lab in labs:
                found.setdefault(lab, ("absent", m.group(0), "normal-phrase rule"))
    return found


In [ ]:
%%writefile mediosis/xr_vision.py
"""Chest X-ray analysis: pretrained TorchXRayVision DenseNet-121 (trained on 8 public datasets:
NIH, CheXpert, MIMIC-CXR, PadChest, RSNA, OpenI, Google, Kaggle) + test-time-augmentation
uncertainty + Grad-CAM localisation."""
import matplotlib
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchxrayvision as xrv
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model = None


def model():
    global _model
    if _model is None:
        _model = xrv.models.DenseNet(weights="densenet121-res224-all").to(DEVICE).eval()
    return _model


def preprocess(img):
    """PIL / ndarray -> (1,1,224,224) tensor in xrv's [-1024,1024] range, plus 224px display image."""
    if isinstance(img, Image.Image):
        img = np.array(img.convert("L"))
    elif img.ndim == 3:
        img = np.array(Image.fromarray(img).convert("L"))
    arr = xrv.datasets.normalize(img.astype(np.float32), 255)[None]
    arr = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(224)])(arr)
    disp = (arr[0] + 1024) / 2048
    return torch.from_numpy(arr)[None].float().to(DEVICE), np.clip(disp, 0, 1)


def _labels():
    return [p for p in model().pathologies if p]


@torch.no_grad()
def predict(x, n_tta=8, seed=0):
    """Returns {pathology: (mean_prob, std)}. 0.5 = calibrated operating threshold of the model."""
    m = model()
    g = torch.Generator(device="cpu").manual_seed(seed)
    aug = torchvision.transforms.RandomAffine(degrees=5, translate=(0.04, 0.04), scale=(0.95, 1.05))
    batch = [x[0]]
    for _ in range(n_tta - 1):
        torch.manual_seed(int(torch.randint(0, 10**6, (1,), generator=g)))
        batch.append(aug(x[0]))
    probs = m(torch.stack(batch)).cpu().numpy()  # (n_tta, 18)
    return {p: (float(probs[:, i].mean()), float(probs[:, i].std()))
            for i, p in enumerate(m.pathologies) if p}


def gradcam(x, pathology):
    """Grad-CAM on the last dense block for one pathology. Returns heatmap (224,224) in [0,1]."""
    m = model()
    idx = list(m.pathologies).index(pathology)
    x = x.clone().requires_grad_(False)
    feats = m.features(x)
    feats.retain_grad()
    pooled = F.adaptive_avg_pool2d(F.relu(feats), 1).flatten(1)
    logit = m.classifier(pooled)[0, idx]
    m.zero_grad()
    logit.backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * feats).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


def overlay(disp, cam, alpha=0.45):
    heat = matplotlib.colormaps["jet"](cam)[..., :3]
    base = np.stack([disp] * 3, -1)
    return Image.fromarray((255 * ((1 - alpha) * base + alpha * heat)).astype(np.uint8))


In [ ]:
%%writefile train.py
"""Train + evaluate every Mediosis model and write artifacts/ and results/metrics.md.

    python train.py            # ~2-4 min on Colab T4, ~3 min on CPU
"""
import json
import os
import time

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

from mediosis import rx_correct, rx_ner, xr_report
from mediosis.rx_realtest import REAL_DRUG_GOLD, real_test_set

ART, RES = "artifacts", "results"
os.makedirs(ART, exist_ok=True)
os.makedirs(RES, exist_ok=True)
t0 = time.time()
M = {}

print("1/3  fastText subword model + drug-name corrector")
ft = rx_correct.train_fasttext()
ft.save(f"{ART}/fasttext.model")
corr = rx_correct.DrugCorrector(ft)
M["corrector_synthetic_1-2_edits"], n = rx_correct.evaluate(corr)
M["corrector_synthetic_3_edits"], _ = rx_correct.evaluate(corr, n_edits=3, seed=321)
M["corrector_real_style"], _ = rx_correct.evaluate(corr, tests=[(k, v) for k, v in REAL_DRUG_GOLD.items()])
print("   ", {k: v for k, v in M.items()})

print("2/3  BiLSTM-CRF prescription NER")
ner = rx_ner.train_ner(ft)
rx_ner.save(ner, f"{ART}/ner.pt")
f1_syn, rep_syn = rx_ner.evaluate_ner(ner, ft)
real = real_test_set()
from seqeval.metrics import classification_report, f1_score
pred = rx_ner.predict(ner, ft, [t for t, _ in real])
f1_real = f1_score([g for _, g in real], pred)
rep_real = classification_report([g for _, g in real], pred, digits=3, zero_division=0)
M["ner_f1_synthetic_heavy_noise"], M["ner_f1_real_style"] = f1_syn, f1_real
print(f"    F1 synthetic={f1_syn:.3f}  real-style={f1_real:.3f}")

print("3/3  Report assertion classifier (BiGRU + self-attention)")
model, vocab = xr_report.train_assertion(verbose=2)
asserter = xr_report.Asserter(model, vocab)
asserter.save(ART)
M["assertion"] = xr_report.evaluate_assertion(asserter)

json.dump(M, open(f"{RES}/metrics.json", "w"), indent=2, default=float)

# ---------- markdown table ----------
L = ["# Mediosis — evaluation results", "",
     "## RxDecode · drug-name correction (top-1 accuracy)", "",
     "| Test set | Edit distance only | fastText only | **Ensemble** |", "|---|---|---|---|"]
for k, name in [("corrector_synthetic_1-2_edits", f"Synthetic OCR noise, 1–2 edits (n={n})"),
                ("corrector_synthetic_3_edits", "Synthetic OCR noise, 3 edits (harder)"),
                ("corrector_real_style", f"Hand-written real-style misreads (n={len(REAL_DRUG_GOLD)})")]:
    r = M[k]
    L.append(f"| {name} | {r['fuzzy']:.3f} | {r['fasttext']:.3f} | **{r['ensemble']:.3f}** |")
L += ["", "## RxDecode · BiLSTM-CRF NER (entity-level F1)", "",
      "| Test set | F1 |", "|---|---|",
      f"| Synthetic, unseen seed, heavier OCR noise (n=2000 lines) | {f1_syn:.3f} |",
      f"| Hand-written real-style lines, unseen formats (n={len(real)} lines) | {f1_real:.3f} |",
      "", "```", rep_real.strip(), "```",
      "", "## XCheck · report assertion (present / absent / uncertain)", "",
      "| Test set | NegEx rules acc / macro-F1 | BiGRU+self-attn acc / macro-F1 |", "|---|---|---|"]
for name, r in M["assertion"].items():
    a, b = r["NegEx rules"], r["BiGRU+self-attn (+rule fallback)"]
    L.append(f"| {name} | {a[0]:.3f} / {a[1]:.3f} | {b[0]:.3f} / {b[1]:.3f} |")
L += ["", "## XCheck · image model", "",
      "Pretrained TorchXRayVision DenseNet-121 (`densenet121-res224-all`), published AUROCs in "
      "Cohen et al., *TorchXRayVision* (MIDL 2022). Not re-trained here.", "",
      "## Honest caveats", "",
      "- Synthetic test sets come from generators we wrote; real-world accuracy will be lower.",
      "- Real-style sets are small (tens of examples) — treat as sanity checks, not clinical validation.",
      "- Not a medical device. Not clinically validated. For education/research demos only.",
      "", f"_Training time: {time.time() - t0:.0f}s_"]
open(f"{RES}/metrics.md", "w").write("\n".join(L))
print("\n".join(L))


In [ ]:
%%writefile app.py
"""Mediosis — Gradio demo.   python app.py  (add --share for a public link)"""
import os
import sys

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import gradio as gr
import pandas as pd
from gensim.models import FastText
from PIL import Image

from mediosis import rx_ner, xr_vision
from mediosis.rx_correct import DrugCorrector
from mediosis.rx_ocr import read_prescription
from mediosis.rx_pipeline import decode_lines, explain
from mediosis.xr_check import cross_check
from mediosis.xr_report import Asserter

ART = "artifacts"
if not os.path.exists(f"{ART}/ner.pt"):
    print("No trained artifacts found -> running train.py first")
    os.system(f"{sys.executable} train.py")

FT = FastText.load(f"{ART}/fasttext.model")
CORR = DrugCorrector(FT)
NER = rx_ner.load(f"{ART}/ner.pt")
ASSERT = Asserter.load(ART)
xr_vision.model()  # warm-up download

DISCLAIMER = ("⚠️ **Research / education demo — not a medical device, not clinically validated.** "
              "Never change medication or act on a scan based on this tool. Always consult a doctor or pharmacist.")


# ---------------- RxDecode ----------------
def run_rx(image, typed, handwriting, lang):
    ocr_rows = []
    if typed and typed.strip():
        lines = [l for l in typed.splitlines() if l.strip()]
    elif image is not None:
        res = read_prescription(image, handwriting=handwriting)
        lines = [t for t, _ in res]
        ocr_rows = [{"OCR line": t, "OCR conf": round(c, 2)} for t, c in res]
    else:
        return "Upload a prescription image or type/paste the lines.", None, None, None
    meds, alerts = decode_lines(lines, NER, FT, CORR)
    if not meds:
        return "No medicines detected.", pd.DataFrame(ocr_rows), None, None
    table = pd.DataFrame([{"OCR word": m["ocr_word"], "Matched drug": m["drug"], "Match conf": m["match_conf"],
                           "Generic (ask for Jan Aushadhi)": m["generic"], "Class": m["drug_class"],
                           "Strength": m["strength"], "How often": m["freq_en"], "Food": m["instr_en"],
                           "Duration": m["dur_en"]} for m in meds])
    al = pd.DataFrame(alerts) if alerts else pd.DataFrame([{"severity": "—", "drugs": "—",
                                                            "issue": "No interactions found in Mediosis' rule base (not exhaustive)."}])
    md = f"### {'दवा कैसे लें' if lang == 'हिन्दी' else 'How to take your medicines'}\n" + \
         explain(meds, "hi" if lang == "हिन्दी" else "en")
    if alerts:
        md += f"\n\n### ⚠️ {len(alerts)} safety alert(s) — show this to your doctor/pharmacist"
    return md, pd.DataFrame(ocr_rows) if ocr_rows else None, table, al


# ---------------- XCheck ----------------
def run_xr(image, report):
    if image is None:
        return "Upload a chest X-ray (frontal PA/AP view).", None, None
    x, disp = xr_vision.preprocess(image)
    probs = xr_vision.predict(x)
    rows, _ = cross_check(probs, report, ASSERT)
    df = pd.DataFrame(rows)[["flag", "pathology", "image_prob", "tta_std", "image", "report", "evidence", "engine"]] \
        if rows else pd.DataFrame()
    top = sorted(probs.items(), key=lambda kv: -kv[1][0])[:3]
    flagged = [r["pathology"] for r in rows if r["severity"] >= 2]
    targets = list(dict.fromkeys(flagged + [k for k, _ in top]))[:4]
    gallery = [(xr_vision.overlay(disp, xr_vision.gradcam(x, t)), f"{t}  p={probs[t][0]:.2f}") for t in targets]
    n_red = sum(r["severity"] == 3 for r in rows)
    n_or = sum(r["severity"] == 2 for r in rows)
    summary = (f"### Cross-check: {n_red} contradiction(s), {n_or} possible miss/over-call(s)\n"
               f"Image model: TorchXRayVision DenseNet-121 · 0.5 = calibrated operating point · "
               f"uncertainty from 8× test-time augmentation.\n\n"
               + ("" if report and report.strip() else "_No report given — showing image findings only._"))
    return summary, df, gallery


DEMO_REPORT = ("Heart size is normal. No pleural effusion or pneumothorax. "
               "Lungs are clear. No acute cardiopulmonary process.")

with gr.Blocks(title="Mediosis", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 Mediosis\n**Prescription decoding + radiology second-reader** · by Genzis\n\n" + DISCLAIMER)
    with gr.Tab("💊 RxDecode — prescription reader"):
        with gr.Row():
            with gr.Column():
                rx_img = gr.Image(type="pil", label="Prescription photo")
                rx_txt = gr.Textbox(lines=5, label="…or type / paste prescription lines (overrides image)")
                with gr.Row():
                    hw = gr.Checkbox(label="Handwritten (TrOCR re-read, slower)")
                    lang = gr.Radio(["English", "हिन्दी"], value="English", label="Explain in")
                rx_btn = gr.Button("Decode prescription", variant="primary")
                gr.Examples([["samples/prescription_demo.png", "", False, "English"],
                             [None, "Tab Glycornet 500mg 1-0-1 after food x 30 days\nTab Ciplox 500 BD x 5 days\n"
                                    "Tab Shelca1 500 OD after lunch\nTab Thyronorm 50mcg 1-0-0 empty stomach", False, "हिन्दी"]],
                            [rx_img, rx_txt, hw, lang])
            with gr.Column():
                rx_md = gr.Markdown()
                rx_alerts = gr.Dataframe(label="Interaction & duplicate-drug alerts", wrap=True)
        rx_table = gr.Dataframe(label="Decoded medicines", wrap=True)
        rx_ocr = gr.Dataframe(label="Raw OCR lines")
        rx_btn.click(run_rx, [rx_img, rx_txt, hw, lang], [rx_md, rx_ocr, rx_table, rx_alerts])

    with gr.Tab("🫁 XCheck — X-ray vs report"):
        with gr.Row():
            with gr.Column():
                xr_img = gr.Image(type="pil", label="Chest X-ray")
                xr_rep = gr.Textbox(lines=5, label="Radiologist's report (optional)")
                xr_btn = gr.Button("Cross-check", variant="primary")
                gr.Examples([["samples/00000001_000.png", DEMO_REPORT],
                             ["samples/16747_3_1.jpg", "Small left upper lobe nodule. No effusion."]], [xr_img, xr_rep])
            with gr.Column():
                xr_md = gr.Markdown()
                xr_gal = gr.Gallery(label="Grad-CAM — where the model looked", columns=2, height=420)
        xr_df = gr.Dataframe(label="Finding-by-finding cross-check", wrap=True)
        xr_btn.click(run_xr, [xr_img, xr_rep], [xr_md, xr_df, xr_gal])

    with gr.Tab("📊 Model card"):
        gr.Markdown(open("results/metrics.md").read() if os.path.exists("results/metrics.md") else "Run train.py")

if __name__ == "__main__":
    demo.launch(share="--share" in sys.argv, debug="--debug" in sys.argv)


In [ ]:
%%writefile make_samples.py
"""Create a demo printed prescription image (with deliberate interactions) in samples/."""
from PIL import Image, ImageDraw, ImageFont


def font(size, bold=False):
    for p in [f"/usr/share/fonts/truetype/dejavu/DejaVuSans{'-Bold' if bold else ''}.ttf",
              f"/usr/share/fonts/truetype/liberation/LiberationSans-{'Bold' if bold else 'Regular'}.ttf"]:
        try:
            return ImageFont.truetype(p, size)
        except OSError:
            pass
    return ImageFont.load_default(size=size)


def make_rx(path="samples/prescription_demo.png"):
    W, H = 1000, 760
    im = Image.new("RGB", (W, H), "white")
    d = ImageDraw.Draw(im)
    d.rectangle([0, 0, W, 110], fill=(225, 238, 250))
    d.text((40, 22), "Dr. A. Sharma, MBBS, MD (Medicine)", fill=(20, 40, 90), font=font(30, True))
    d.text((40, 66), "City Care Clinic  |  Reg. No. 00000 (DEMO)", fill=(60, 60, 60), font=font(20))
    d.text((40, 135), "Patient: Demo Patient   Age: 58   Date: 20/11/2026", fill="black", font=font(22))
    d.text((40, 185), "Rx", fill=(20, 40, 90), font=font(40, True))
    lines = ["1. Tab Crocin 650mg 1-0-1 after food x 5 days",
             "2. Tab Combiflam SOS for pain",
             "3. Tab Warf 5mg OD x 30 days",
             "4. Tab Ecosprin 75mg HS x 30 days",
             "5. Cap Pantocid 40mg OD before breakfast x 14 days",
             "6. Tab Azithral 500mg OD x 3 days",
             "7. Tab Ondem 4mg TDS before food x 3 days"]
    y = 250
    for ln in lines:
        d.text((70, y), ln, fill="black", font=font(26))
        y += 58
    d.text((620, 690), "Signature (demo only)", fill=(90, 90, 90), font=font(18))
    im.save(path)
    return path


if __name__ == "__main__":
    print(make_rx())


In [ ]:
%%writefile requirements.txt
# torch, torchvision, tensorflow/keras, numpy, pandas, scikit-learn, matplotlib come preinstalled on Colab
torch
torchvision
tensorflow>=2.16
keras>=3
torchxrayvision>=1.2
gensim>=4.3.3
pytorch-crf>=0.7.2
rapidfuzz>=3
seqeval>=1.2.2
easyocr>=1.7
transformers>=4.40   # only for the optional handwriting (TrOCR) mode
gradio>=4.44
pandas
scikit-learn
matplotlib
pillow


In [ ]:
%%writefile .gitignore
artifacts/
__pycache__/
*.pyc
.ipynb_checkpoints/
flagged/
.gradio/


In [ ]:
%%writefile README.md
# 🩺 Mediosis

**Prescription decoding + radiology second-reader, built with classic NLP and deep learning.**
A Genzis project · [Open in Colab](#run-in-colab)

> ⚠️ Research / education demo. Not a medical device and not clinically validated. Don't change medication or act on a scan because of this tool.

Mediosis has two modules:

| Module | What it does | Core tech |
|---|---|---|
| 💊 **RxDecode** | Photo of a prescription → reads every medicine, fixes OCR-garbled drug names, explains dose/frequency/duration in **English or Hindi**, flags **drug interactions and duplicate ingredients** (e.g. Crocin + Combiflam = double paracetamol) and gives the generic name to ask for at Jan Aushadhi | EasyOCR (+ optional TrOCR for handwriting) · **fastText subword** embeddings · **BiLSTM-CRF** NER · rule-based interaction engine |
| 🫁 **XCheck** | Chest X-ray + the radiologist's report → finds 18 conditions in the image, reads the report (including negation like *"no effusion"*), and **flags contradictions and possible missed findings**, with Grad-CAM heatmaps | TorchXRayVision DenseNet-121 · test-time-augmentation uncertainty · **Grad-CAM** · **BiGRU + self-attention** assertion classifier vs a NegEx baseline |

## Why it's different

- **OCR-noise-aware drug matching.** OCR turns *Azithromycin* into *Azithrornycin* (`m`→`rn`). fastText builds word vectors from character n-grams, so a misspelt word still lands close to the real one. We train it on prescription text plus simulated OCR corruptions, then combine its cosine similarity with edit distance.
- **Second reader, not a diagnosis.** XCheck doesn't try to replace the radiologist. It flags places where the image model and the written report *disagree*. That's the kind of safety net hospitals actually use.
- **Uncertainty built in.** The image is scored under 8 small augmentations. If the predictions disagree, the result is marked *unstable* instead of giving a confident wrong answer.

## Architecture

```
RxDecode   image ─► EasyOCR ─► line grouping ─► OCR clean-up ─► tokenizer
                                                                   │
                     fastText subword vectors + shape features ◄───┘
                                     │
                         BiLSTM ─► CRF  (DRUG/STRENGTH/FORM/FREQ/INSTR/DUR)
                                     │
             drug corrector (fastText cos ⊕ edit distance) ─► brand → generic
                                     │
              interaction + duplicate rules ─► EN / हिन्दी explanation

XCheck     X-ray ─► DenseNet-121 (8× TTA) ─► probs ± std ─► Grad-CAM
           report ─► sentence split ─► finding mentions ─► mask target (tgt/oth)
                     ─► pre-pad ─► Embedding ─► SpatialDropout1D ─► BiGRU(recurrent_dropout)
                     ─► Multi-head self-attention ─► softmax {present, absent, uncertain}
           image status × report status ─► 🔴 contradiction / 🟠 missed / ✅ consistent
```

## Results

See [`results/metrics.md`](results/metrics.md). It's regenerated every time you run `train.py`.

| | Metric | Score |
|---|---|---|
| Drug-name correction, synthetic OCR noise (n=1780) | top-1 acc (ensemble) | ~0.99 |
| Drug-name correction, heavy noise (3 edits) | top-1 acc (ensemble) | ~0.93 |
| Rx NER, hand-written real-style lines (unseen formats) | entity F1 | ~0.93 |
| Report assertion, hand-written real-style sentences (n=30) | accuracy | 0.90 (NegEx baseline 0.90) |

**Honest caveats:** the synthetic test sets come from our own generators, and the real-style sets are small. These are sanity checks, not clinical validation. The image model is pretrained ([Cohen et al., TorchXRayVision, MIDL 2022](https://github.com/mlmed/torchxrayvision)) and was not re-trained here.

## Run in Colab

1. Open `Mediosis_Colab.ipynb` in Colab → *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. It installs packages, trains everything (~2–3 min), evaluates, and prints a public **gradio.live** link.

## Run locally

```bash
pip install -r requirements.txt
python make_samples.py
python train.py          # writes artifacts/ and results/metrics.md
python app.py --share    # --share gives a public link
```

## Project layout

```
mediosis/
  lexicon.py       drug lexicon, interaction rules, Rx shorthand (OD/BD/TDS/1-0-1…)
  rx_data.py       tokenizer, OCR-noise simulator, synthetic BIO-tagged prescriptions
  rx_correct.py    fastText subword drug-name corrector + eval
  rx_ner.py        BiLSTM-CRF tagger
  rx_ocr.py        EasyOCR + optional TrOCR handwriting
  rx_pipeline.py   end-to-end prescription decoding + EN/HI explanation
  rx_realtest.py   hand-written real-style test lines
  xr_vision.py     DenseNet-121, TTA uncertainty, Grad-CAM
  xr_report.py     report NLP: mention finder, NegEx rules, BiGRU+self-attention classifier
  xr_check.py      image ↔ report cross-check logic
train.py · app.py · make_samples.py
```

## Roadmap

- Real prescription dataset with doctor annotations; fine-tune TrOCR on Indian handwriting
- Replace the rule-based interaction list with DDInter / DrugBank
- Validate XCheck on MIMIC-CXR report–image pairs (needs credentialed access)
- Generate radiology reports with a CNN encoder + attention decoder (IU X-Ray)

## Credits

Built by **Tanush Bhardwaj** ([@tanush69lucifer](https://github.com/tanush69lucifer)) · Genzis — *Generations Ahead*.
Sample X-rays come from the TorchXRayVision test set (NIH ChestX-ray14 / PadChest).


## 2 · Install dependencies (~1–2 min)

In [ ]:
!pip -q install torchxrayvision gensim pytorch-crf rapidfuzz seqeval easyocr gradio
!curl -sSfL -o samples/00000001_000.png https://raw.githubusercontent.com/mlmed/torchxrayvision/master/tests/00000001_000.png
!curl -sSfL -o samples/16747_3_1.jpg https://raw.githubusercontent.com/mlmed/torchxrayvision/master/tests/16747_3_1.jpg
!python make_samples.py && ls samples

## 3 · Train + evaluate (fastText → BiLSTM-CRF → BiGRU self-attention) ~2–3 min

In [ ]:
!python train.py 2>&1 | grep -vE 'step -|oneDNN|cuda_|absl|E0000|W0000|I0000'

In [ ]:
from IPython.display import Markdown
Markdown(open('results/metrics.md').read())

## 4 · Quick headless test

In [ ]:
import os, sys
os.environ['TF_FORCE_GPU_ALLOW_GROWTH']='true'
sys.path.insert(0,'/content/mediosis')
import app
from PIL import Image
from IPython.display import display, Markdown
md_, ocr, table, alerts = app.run_rx(Image.open('samples/prescription_demo.png'), '', False, 'हिन्दी')
display(Markdown(md_)); display(table); display(alerts)
s, df, gal = app.run_xr(Image.open('samples/00000001_000.png'), app.DEMO_REPORT)
display(Markdown(s)); display(df.head(8)); display(gal[0][0])

## 5 · Launch the web app (public link)

In [ ]:
app.demo.launch(share=True, debug=False)

## 6 · Push to GitHub
Create a token at github.com → Settings → Developer settings → **Personal access tokens (classic)** with `repo` scope. Paste it when asked; it's not saved.

In [ ]:
import getpass, subprocess, json, urllib.request
GH_USER = 'tanush69lucifer'  #@param {type:'string'}
REPO = 'mediosis'  #@param {type:'string'}
EMAIL = 'tanushbharadwaj2514@gmail.com'  #@param {type:'string'}
token = getpass.getpass('GitHub token: ')
req = urllib.request.Request('https://api.github.com/user/repos', method='POST',
    data=json.dumps({'name': REPO, 'description': 'Mediosis — prescription decoding + radiology second-reader (NLP + DL)', 'private': False}).encode(),
    headers={'Authorization': f'token {token}', 'Accept': 'application/vnd.github+json'})
try:
    urllib.request.urlopen(req); print('Repo created')
except Exception as e:
    print('Repo exists or could not be created:', e)
cmds = f'''cd /content/mediosis && git init -q 2>/dev/null; git config user.name "{GH_USER}" && git config user.email "{EMAIL}" \\
&& git add -A && (git commit -qm "Mediosis: RxDecode + XCheck with training, eval and Gradio app" || true) && git branch -M main \\
&& git remote remove origin 2>/dev/null; git remote add origin https://{token}@github.com/{GH_USER}/{REPO}.git && git push -u origin main -f'''
r = subprocess.run(cmds, shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
subprocess.run(f'cd /content/mediosis && git remote set-url origin https://github.com/{GH_USER}/{REPO}.git', shell=True)
print(f'👉 https://github.com/{GH_USER}/{REPO}')